# ⚙️ Feature Engineering
Building ML-ready features from raw production data.  
Covers: lag features · rolling statistics · domain-specific petroleum features · cyclical encoding
---

In [ ]:
import sys; sys.path.insert(0, '..')
from src.features import prepare_forecasting_data
import pandas as pd; import numpy as np; import matplotlib.pyplot as plt

data = prepare_forecasting_data()
field = data['field']
print(f"Field shape: {field.shape}")
print(f"\nAll features:")
for i, col in enumerate(field.columns, 1):
    print(f"  {i:2d}. {col}")

## 1. Lag Features
Autoregressive features let the model 'see' past production values.

In [ ]:
lag_cols = [c for c in field.columns if 'lag' in c]
print(f"Lag features ({len(lag_cols)}): {lag_cols[:5]}...")
field[['date','oil_vol'] + lag_cols[:5]].head(10)

## 2. Rolling Statistics

In [ ]:
roll_cols = [c for c in field.columns if 'roll' in c]
print(f"Rolling features ({len(roll_cols)}): {roll_cols}")
# Visualise rolling means
fig, ax = plt.subplots(figsize=(13,5))
ax.plot(field['date'], field['oil_vol'], alpha=0.4, color='steelblue', label='Daily')
ax.plot(field['date'], field['oil_vol_roll_mean7'],  color='orange',  lw=2, label='7d MA')
ax.plot(field['date'], field['oil_vol_roll_mean30'], color='red',     lw=2, label='30d MA')
ax.plot(field['date'], field['oil_vol_roll_mean90'], color='purple',  lw=2, label='90d MA')
ax.set_ylabel('Oil Vol (Sm³/day)'); ax.set_title('Rolling Mean Features', fontsize=13)
ax.legend(); plt.tight_layout(); plt.show()

## 3. Cyclical Encoding (Day of Year)

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(11,4))
axes[0].scatter(field['day_of_year'], field['oil_vol'], alpha=0.1, s=5, c='steelblue')
axes[0].set_xlabel('Day of Year'); axes[0].set_ylabel('Oil Vol'); axes[0].set_title('Raw DOY')
axes[1].scatter(field['doy_sin'], field['doy_cos'], alpha=0.15, s=5, c='orange')
axes[1].set_xlabel('sin(DOY)'); axes[1].set_ylabel('cos(DOY)'); axes[1].set_title('Cyclical Encoding')
plt.suptitle('Day-of-Year: Raw vs Cyclical', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. LSTM Sequence Shape

In [ ]:
print(f"LSTM X_train : {data['lstm']['X_train'].shape}  (samples, timesteps, features)")
print(f"LSTM X_test  : {data['lstm']['X_test'].shape}")
print(f"Sequence length: 30 days → predict day 31")